# RoadWatch – 01 Baseline (YOLOv8n)

Baseline road-damage detector trained on Manasvi's group-aware dataset (`RoadWatch_YOLO_Grouped`).

| ID | Class |
|---|---|
| 0 | D00 – Longitudinal crack |
| 1 | D10 – Transverse crack |
| 2 | D20 – Alligator crack |
| 3 | D40 – Pothole |

**Before you start:** *Runtime → Change runtime type → T4 GPU*, then run the cells from top to bottom.

Training runs are saved to Google Drive (`MyDrive/RoadWatch/runs`), so they survive a Colab disconnect.

In [1]:
!rm -rf /content/drive/MyDrive/RoadWatch/runs/v8n_baseline

## a) Install packages and mount Google Drive

In [2]:
# Ultralytics provides YOLOv8; gdown downloads files from Google Drive links.
!pip install -q ultralytics gdown

# Mount Drive so training results are stored persistently.
from google.colab import drive
drive.mount('/content/drive')

# Check that a GPU is attached (should list a Tesla T4).
!nvidia-smi -L

Mounted at /content/drive
GPU 0: Tesla T4 (UUID: GPU-b7ab297c-5a71-b155-6261-ead1a277012a)


## b) Download and unzip the dataset

> **If `gdown` fails** (quota exceeded / permission error): open the shared file in Google Drive,
> choose **"Add shortcut to Drive"**, and run the fallback cell below instead. It copies the zip from
> `/content/drive/MyDrive`.

In [3]:
# Download the grouped dataset zip from Google Drive.
!gdown 1kIe0dMfRJWRbKxM2ZYB4_9GiU8pCYs7F -O /content/RoadWatch_YOLO_Grouped.zip

# Unzip into /content/data (-q quiet, -o overwrite if re-running).
!unzip -q -o /content/RoadWatch_YOLO_Grouped.zip -d /content/data

Downloading...
From (original): https://drive.google.com/uc?id=1kIe0dMfRJWRbKxM2ZYB4_9GiU8pCYs7F
From (redirected): https://drive.google.com/uc?id=1kIe0dMfRJWRbKxM2ZYB4_9GiU8pCYs7F&confirm=t&uuid=0a421a2a-4cba-4383-9ce7-cd394c392eec
To: /content/RoadWatch_YOLO_Grouped.zip
100% 774M/774M [00:09<00:00, 79.4MB/s]


In [4]:
# FALLBACK – only if gdown failed. Adjust the path if the shortcut is in a sub-folder of MyDrive.
# !cp "/content/drive/MyDrive/RoadWatch_YOLO_Grouped.zip" /content/RoadWatch_YOLO_Grouped.zip
# !unzip -q -o /content/RoadWatch_YOLO_Grouped.zip -d /content/data

## c) Find the dataset root and check the splits

The zip may contain an extra top-level folder, so we search for the folder that contains `images/train`.
Expected image counts (from the dataset README): **train ≈ 5001, val ≈ 1429, test ≈ 714**.

In [5]:
from pathlib import Path

DATA_DIR = Path('/content/data')
IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.bmp'}
EXPECTED = {'train': 5001, 'val': 1429, 'test': 714}

# The dataset root is the folder whose images/train sub-folder exists.
candidates = [p.parent.parent for p in DATA_DIR.rglob('images/train') if p.is_dir()]
if not candidates:
    raise FileNotFoundError(f'No images/train folder found under {DATA_DIR}. Did the unzip work?')
DATASET_ROOT = min(candidates, key=lambda p: len(p.parts))  # shallowest match
print('Dataset root:', DATASET_ROOT)

for split in ['train', 'val', 'test']:
    images = [f for f in (DATASET_ROOT / 'images' / split).iterdir() if f.suffix.lower() in IMAGE_EXTS]
    labels = list((DATASET_ROOT / 'labels' / split).glob('*.txt'))
    empty = sum(1 for f in labels if f.stat().st_size == 0)  # negative images have empty label files
    status = 'OK' if len(images) == EXPECTED[split] else f'WARNING: expected about {EXPECTED[split]}'
    print(f'{split:5s}: {len(images):5d} images | {len(labels):5d} labels ({empty} empty = negatives) | {status}')

Dataset root: /content/data/RoadWatch_YOLO_Grouped
train:  5001 images |  5001 labels (2745 empty = negatives) | OK
val  :  1429 images |  1429 labels (784 empty = negatives) | OK
test :   714 images |   714 labels (392 empty = negatives) | OK


## d) Write `data.yaml`

In [6]:
import yaml

DATA_YAML = '/content/data.yaml'
data_config = {
    'path': str(DATASET_ROOT),
    'train': 'images/train',
    'val': 'images/val',
    'test': 'images/test',
    'nc': 4,
    'names': ['D00', 'D10', 'D20', 'D40'],
}
with open(DATA_YAML, 'w') as f:
    yaml.safe_dump(data_config, f, sort_keys=False)

print(open(DATA_YAML).read())

path: /content/data/RoadWatch_YOLO_Grouped
train: images/train
val: images/val
test: images/test
nc: 4
names:
- D00
- D10
- D20
- D40



## e) Train the YOLOv8n baseline

About 50 epochs on a T4. Early stopping (`patience=15`) ends training if validation mAP stops improving.
If a folder `v8n_baseline` already exists, Ultralytics saves to `v8n_baseline2`, `v8n_baseline3`, …

In [7]:
from ultralytics import YOLO

PROJECT = '/content/drive/MyDrive/RoadWatch/runs'
RUN_NAME = 'v8n_baseline'

model = YOLO('yolov8n.pt')  # COCO-pretrained starting point
results = model.train(
    data=DATA_YAML,
    epochs=50,
    imgsz=640,
    batch=16,
    seed=42,
    patience=15,
    project=PROJECT,
    name=RUN_NAME,
)
RUN_DIR = Path(results.save_dir)  # actual folder (may be v8n_baseline2 etc.)
print('Run saved to:', RUN_DIR)

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
Ultralytics 8.4.216 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None,

## f) Resume after a Colab disconnect

If the runtime disconnects mid-training:

1. Reconnect (T4 GPU) and re-run cells **a**, **b**, **c** and **d**. The resumed run reads the dataset
   from `/content/data.yaml`, so the dataset must be unzipped and the yaml re-created first.
2. Run the cell below. It continues from `last.pt` with the original settings (epochs, seed, …).

Change `RUN_NAME` if your run was saved as `v8n_baseline2`, etc.

In [8]:
from pathlib import Path
from ultralytics import YOLO

PROJECT = '/content/drive/MyDrive/RoadWatch/runs'
RUN_NAME = 'v8n_baseline'
last_pt = Path(PROJECT) / RUN_NAME / 'weights' / 'last.pt'

if last_pt.exists():
    model = YOLO(str(last_pt))
    results = model.train(resume=True)  # continues the same run in the same folder
    RUN_DIR = Path(results.save_dir)
else:
    print(f'No checkpoint at {last_pt} – nothing to resume.')

No checkpoint at /content/drive/MyDrive/RoadWatch/runs/v8n_baseline/weights/last.pt – nothing to resume.


## g) Evaluate `best.pt` on the test split

The test split was not used for training or model selection, so these are the numbers to report.

In [11]:
!ls /content/drive/MyDrive/RoadWatch/runs/
!ls -la /content/drive/MyDrive/RoadWatch/runs/*/weights/

v8n_baseline  v8n_baseline-2
/content/drive/MyDrive/RoadWatch/runs/v8n_baseline-2/weights/:
total 12719
drwx------ 2 root root    4096 Nov 23 15:14 .
drwx------ 3 root root    4096 Nov 23 15:14 ..
-rw------- 1 root root 6507946 Nov 23 15:14 best.pt
-rw------- 1 root root 6507946 Nov 23 15:14 last.pt

/content/drive/MyDrive/RoadWatch/runs/v8n_baseline/weights/:
total 8
drwx------ 2 root root 4096 Nov 23 13:11 .
drwx------ 3 root root 4096 Nov 23 13:11 ..


In [15]:
import json
from pathlib import Path
from ultralytics import YOLO

PROJECT = '/content/drive/MyDrive/RoadWatch/runs'
RUN_NAME = 'v8n_baseline-2'
RUN_DIR = Path(PROJECT) / RUN_NAME          # change if your run is v8n_baseline2, etc.
best_pt = RUN_DIR / 'weights' / 'best.pt'

metrics = YOLO(str(best_pt)).val(
    data='/content/data.yaml',
    split='test',
    imgsz=640,
    batch=16,
    project=PROJECT,
    name=f'{RUN_NAME}_test',
)

box = metrics.box
print(f'mAP50     : {box.map50:.4f}')
print(f'mAP50-95  : {box.map:.4f}')
print(f'Precision : {box.mp:.4f}')
print(f'Recall    : {box.mr:.4f}')

# Per-class AP50. ap_class_index lists the classes that appear in the test split, in the same order as ap50.
print('\nPer-class AP50:')
per_class = {}
for i, class_id in enumerate(box.ap_class_index):
    name = metrics.names[int(class_id)]
    per_class[name] = round(float(box.ap50[i]), 4)
    print(f'  {name}: {box.ap50[i]:.4f}')

# Keep a copy next to the run on Drive (copy it into model_training/reports/ for the repo).
summary = {
    'weights': str(best_pt),
    'split': 'test',
    'mAP50': round(float(box.map50), 4),
    'mAP50_95': round(float(box.map), 4),
    'precision': round(float(box.mp), 4),
    'recall': round(float(box.mr), 4),
    'per_class_AP50': per_class,
}
out = RUN_DIR / 'test_metrics.json'
out.write_text(json.dumps(summary, indent=2))
print('\nSaved', out)

Ultralytics 8.4.216 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 3,133,436 parameters, 0 gradients, 8.6 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1937.1±652.5 MB/s, size: 52.5 KB)
val: Scanning /content/data/RoadWatch_YOLO_Grouped/labels/test... 714 images, 392 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 714/714 2.2Kit/s 0.3s
val: New cache created: /content/data/RoadWatch_YOLO_Grouped/labels/test.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 45/45 5.0it/s 9.1s
                   all        714        681      0.371      0.387      0.341      0.148
                   D00        101        144      0.375      0.299      0.281      0.129
                   D10          8          8      0.173      0.125     0.0396     0.0178
                   D20        190        211      0.509      0.694      0.655      0.303
                   D40        151        318